In [ ]:
''' 
ChromaCRISPR Phase 1

Notebook 01

Data Inventory

Objective:

Verify that all datasets required for Phase 1 Week 2 were successfully acquired.

This notebook performs:

- Inventory of ENCODE epigenomic data
- Inventory of CRISPR datasets
- File integrity checks
- Dataset size summary
- Export of inventory report

No preprocessing is performed here.
'''

In [1]:
from pathlib import Path
import pandas as pd
import pyBigWig

In [2]:
PROJECT_ROOT = Path("..")

ENCODE_DIR = PROJECT_ROOT / "data/raw/encode_k562"

CRISPR_DIR = PROJECT_ROOT / "data/raw/crispr_datasets"

OUTPUT = PROJECT_ROOT / "data/interim"

OUTPUT.mkdir(exist_ok=True)

In [3]:
encode = []

for f in sorted(ENCODE_DIR.glob("*")):

    if f.is_file():

        encode.append(
            {
                "file": f.name,
                "extension": f.suffix,
                "size_MB": round(f.stat().st_size / 1024**2,2)
            }
        )

encode_df = pd.DataFrame(encode)

encode_df

,file,extension,size_MB
0,ENCFF000BXD.bigWig,.bigWig,383.79
1,ENCFF019IPA.bigWig,.bigWig,1607.65
2,ENCFF502GAA.bigWig,.bigWig,558.39
3,ENCFF900IMQ.bigWig,.bigWig,323.42
4,ENCFF949BXN.bigWig,.bigWig,180.38
5,download_log.txt,.txt,0.00


In [4]:
validation = []

for bw in ENCODE_DIR.glob("*.bigWig"):

    try:

        x = pyBigWig.open(str(bw))

        chroms = len(x.chroms())

        x.close()

        validation.append(
            {
                "file": bw.name,
                "status": "OK",
                "chromosomes": chroms
            }
        )

    except Exception as e:

        validation.append(
            {
                "file": bw.name,
                "status": str(e),
                "chromosomes": None
            }
        )

validation_df = pd.DataFrame(validation)

validation_df

,file,status,chromosomes
0,ENCFF502GAA.bigWig,OK,121
1,ENCFF019IPA.bigWig,OK,147
2,ENCFF949BXN.bigWig,OK,137
3,ENCFF000BXD.bigWig,OK,23
4,ENCFF900IMQ.bigWig,OK,25


In [5]:
records = []

for dataset in sorted(CRISPR_DIR.iterdir()):

    if not dataset.is_dir():
        continue

    total_size = 0

    extensions = {}

    nfiles = 0

    for f in dataset.rglob("*"):

        if f.is_file():

            nfiles += 1

            total_size += f.stat().st_size

            ext = f.suffix.lower()

            extensions[ext] = extensions.get(ext,0)+1

    records.append(
        {
            "dataset": dataset.name,
            "files": nfiles,
            "size_MB": round(total_size/1024**2,2),
            "extensions": ", ".join(sorted(extensions.keys()))
        }
    )

crispr_df = pd.DataFrame(records)

crispr_df

,dataset,files,size_MB,extensions
0,Gasperini2019,2,0.85,.xlsx
1,Horlbeck2016,1,1.37,.xlsx
2,Replogle2022,1,1.28,.xlsx
3,Sanson2018,1,9.46,.xlsx


In [6]:
summary = pd.DataFrame(
    {
        "Category":[
            "ENCODE",
            "CRISPR datasets"
        ],
        "Datasets":[
            len(validation_df),
            len(crispr_df)
        ],
        "Total size (GB)":[
            round(
                encode_df["size_MB"].sum()/1024,
                2
            ),
            round(
                crispr_df["size_MB"].sum()/1024,
                2
            )
        ]
    }
)

summary

,Category,Datasets,Total size (GB)
0,ENCODE,5,2.98
1,CRISPR datasets,4,0.01


In [7]:
report = []

report.append("# ChromaCRISPR Phase 1")
report.append("")
report.append("## Week 2 Data Inventory")
report.append("")
report.append(summary.to_markdown(index=False))
report.append("")
report.append("### CRISPR datasets")
report.append("")
report.append(crispr_df.to_markdown(index=False))
report.append("")
report.append("### ENCODE files")
report.append("")
report.append(encode_df.to_markdown(index=False))

with open(
    OUTPUT/"week2_inventory_summary.md",
    "w"
) as f:

    f.write("\n".join(report))

print("Inventory exported.")

Inventory exported.
